# Eurostat Socioeconomic Data Collection

This notebook retrieves socioeconomic indicators from the Eurostat API for each of the 30 European cities in the study. These serve as predictors of light pollution in the regression analysis.

**Panel variables (annual, 2013-2024 where available):**
- Population (Urban Audit cities database)
- GDP per capita in EUR per inhabitant (NUTS-3 regional accounts)

**Cross-sectional variables (latest available year):**
- Population density (NUTS-3 level)

**Output:** `../data/processed/eurostat_city_panel.csv` (360 rows: 30 cities x 12 years)

**Notes:**
- Eurostat Urban Audit uses city codes (e.g., ES001C for Madrid)
- GDP data uses NUTS-3 regional codes (e.g., ES300 for Madrid)
- Norway (Oslo) is EEA, not EU -- data availability may be partial
- Greece uses country code EL in Eurostat (not GR)

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

BASE_URL = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
YEARS = list(range(2013, 2025))
SLEEP_SECONDS = 2

print(f"Target years: {YEARS[0]}-{YEARS[-1]}")

In [ ]:
cities = pd.read_csv("../data/processed/cities.csv")
print(f"Loaded {len(cities)} cities")
cities.head()

## City-to-Eurostat code mapping

Each city is mapped to:
- An **Urban Audit code** for city-level datasets (population)
- A **NUTS-3 code** (or best available NUTS level) for regional datasets (GDP, density)

These codes were verified against the Eurostat API. Special cases:
- **Netherlands:** NUTS-3 GDP data is unavailable; Amsterdam uses NUTS-2 (NL32 = Noord-Holland)
- **Rotterdam:** No GDP at any sub-national level that maps cleanly; uses NUTS-2 NL33 with fallback
- **Lisbon:** NUTS-3 code is PT1A0 (Grande Lisboa), not PT170
- **Riga:** NUTS-3 code is LV00A, not LV006
- **Greece:** Eurostat uses EL prefix (not GR)

In [ ]:
CITY_CODES = {
    "Madrid":     {"urban_audit": "ES001C", "nuts3": "ES300"},
    "Barcelona":  {"urban_audit": "ES002C", "nuts3": "ES511"},
    "Paris":      {"urban_audit": "FR001C", "nuts3": "FR101"},
    "Lyon":       {"urban_audit": "FR003C", "nuts3": "FRK26"},
    "Berlin":     {"urban_audit": "DE001C", "nuts3": "DE300"},
    "Hamburg":    {"urban_audit": "DE002C", "nuts3": "DE600"},
    "Munich":     {"urban_audit": "DE004C", "nuts3": "DE212"},
    "Rome":       {"urban_audit": "IT001C", "nuts3": "ITI43"},
    "Milan":      {"urban_audit": "IT002C", "nuts3": "ITC4C"},
    "Naples":     {"urban_audit": "IT003C", "nuts3": "ITF33"},
    "Amsterdam":  {"urban_audit": "NL002C", "nuts3": "NL32"},   # NUTS-2 fallback
    "Rotterdam":  {"urban_audit": "NL003C", "nuts3": "NL33C"},  # may lack GDP data
    "Brussels":   {"urban_audit": "BE001C", "nuts3": "BE100"},
    "Vienna":     {"urban_audit": "AT001C", "nuts3": "AT130"},
    "Prague":     {"urban_audit": "CZ001C", "nuts3": "CZ010"},
    "Warsaw":     {"urban_audit": "PL001C", "nuts3": "PL911"},
    "Budapest":   {"urban_audit": "HU001C", "nuts3": "HU110"},
    "Copenhagen": {"urban_audit": "DK001C", "nuts3": "DK011"},
    "Stockholm":  {"urban_audit": "SE001C", "nuts3": "SE110"},
    "Oslo":       {"urban_audit": "NO001C", "nuts3": "NO081"},
    "Helsinki":   {"urban_audit": "FI001C", "nuts3": "FI1B1"},
    "Lisbon":     {"urban_audit": "PT001C", "nuts3": "PT1A0"},
    "Porto":      {"urban_audit": "PT002C", "nuts3": "PT11A"},
    "Athens":     {"urban_audit": "EL001C", "nuts3": "EL303"},
    "Dublin":     {"urban_audit": "IE001C", "nuts3": "IE061"},
    "Bucharest":  {"urban_audit": "RO001C", "nuts3": "RO321"},
    "Sofia":      {"urban_audit": "BG001C", "nuts3": "BG411"},
    "Zagreb":     {"urban_audit": "HR001C", "nuts3": "HR050"},
    "Tallinn":    {"urban_audit": "EE001C", "nuts3": "EE001"},
    "Riga":       {"urban_audit": "LV001C", "nuts3": "LV00A"},
}

# Validate mapping covers all cities
assert set(CITY_CODES.keys()) == set(cities["city"]), "Mapping does not cover all cities!"
print(f"Mapping defined for {len(CITY_CODES)} cities")

# Build reverse lookups
ua_to_city = {v["urban_audit"]: k for k, v in CITY_CODES.items()}
nuts_to_city = {v["nuts3"]: k for k, v in CITY_CODES.items()}

## Eurostat API helpers

The Eurostat REST API returns data in JSON-stat format, which uses positional flat indexing across multiple dimensions. The helper below unpacks this into a tidy DataFrame.

In [ ]:
def fetch_eurostat(dataset_code: str, params: list[tuple[str, str]],
                   timeout: int = 60) -> dict | None:
    """
    Fetch data from the Eurostat API.
    
    Parameters
    ----------
    dataset_code : str
        Eurostat dataset code (e.g., 'urb_cpop1').
    params : list of (key, value) tuples
        Query parameters. Use repeated keys for multi-value dimensions.
    
    Returns
    -------
    dict or None
        Raw JSON response, or None on failure.
    """
    url = f"{BASE_URL}/{dataset_code}"
    params_with_lang = params + [("lang", "en")]

    try:
        r = requests.get(url, params=params_with_lang, timeout=timeout)
        if r.status_code == 200:
            return r.json()
        elif r.status_code == 413:
            print(f"  WARNING: Response too large for {dataset_code}. Narrow the query.")
        elif r.status_code == 400:
            err_msg = r.json().get("error", [{}])[0].get("label", "Unknown error")
            print(f"  WARNING: Bad request for {dataset_code}: {err_msg[:120]}")
        else:
            print(f"  WARNING: HTTP {r.status_code} for {dataset_code}")
    except requests.exceptions.Timeout:
        print(f"  WARNING: Timeout for {dataset_code}")
    except Exception as e:
        print(f"  WARNING: Error fetching {dataset_code}: {e}")

    return None


def parse_jsonstat(data: dict, geo_dim: str = "geo") -> pd.DataFrame:
    """
    Parse a Eurostat JSON-stat response into a tidy DataFrame.
    
    The JSON-stat format uses flat positional indexing across dimensions.
    This function decodes each value's position back to dimension labels.
    
    Parameters
    ----------
    data : dict
        Raw JSON response from Eurostat API.
    geo_dim : str
        Name of the geographic dimension ('geo' for NUTS, 'cities' for Urban Audit).
    
    Returns
    -------
    pd.DataFrame
        Tidy DataFrame with one row per observation.
    """
    dims = data["id"]
    sizes = data["size"]
    values = data.get("value", {})

    if not values:
        return pd.DataFrame()

    # Build ordered category lists for each dimension
    dim_cats = {}
    for d in dims:
        idx_map = data["dimension"][d]["category"]["index"]
        # index can be {code: position} -- sort by position to get ordered list
        if isinstance(idx_map, dict):
            sorted_codes = sorted(idx_map.keys(), key=lambda k: idx_map[k])
        else:
            sorted_codes = list(idx_map)
        dim_cats[d] = sorted_codes

    # Decode flat index to dimension positions
    records = []
    for flat_idx_str, val in values.items():
        flat_idx = int(flat_idx_str)
        remaining = flat_idx
        row = {}
        for i in range(len(dims) - 1, -1, -1):
            pos = remaining % sizes[i]
            remaining //= sizes[i]
            row[dims[i]] = dim_cats[dims[i]][pos]
        row["value"] = val
        records.append(row)

    return pd.DataFrame(records)

## Population (Urban Audit)

Dataset `urb_cpop1`, indicator `DE1001V` (total population on 1 January). Queried for all 30 Urban Audit city codes at once.

In [ ]:
# Build batch params for all 30 cities
pop_params = [("indic_ur", "DE1001V"),
              ("sinceTimePeriod", str(YEARS[0])),
              ("untilTimePeriod", str(YEARS[-1]))]
for city_name in CITY_CODES:
    pop_params.append(("cities", CITY_CODES[city_name]["urban_audit"]))

pop_raw = fetch_eurostat("urb_cpop1", pop_params)

if pop_raw:
    df_pop = parse_jsonstat(pop_raw, geo_dim="cities")
    # Map city codes back to city names
    df_pop["city"] = df_pop["cities"].map(ua_to_city)
    df_pop["year"] = df_pop["time"].astype(int)
    df_pop = df_pop.rename(columns={"value": "population"})
    df_pop = df_pop[["city", "year", "population"]].copy()
    print(f"Population data: {len(df_pop)} observations")
    print(f"Cities with data: {df_pop['city'].nunique()}/30")
    print(f"Years covered: {df_pop['year'].min()}-{df_pop['year'].max()}")
else:
    print("ERROR: Population query failed.")
    df_pop = pd.DataFrame(columns=["city", "year", "population"])

In [ ]:
# Coverage matrix: which cities have data for which years
if len(df_pop) > 0:
    coverage = df_pop.pivot_table(index="city", columns="year",
                                  values="population", aggfunc="count")
    coverage = coverage.reindex(columns=YEARS).fillna(0).astype(int)
    missing_cities = set(cities["city"]) - set(df_pop["city"])
    print(f"Cities with ZERO population data: {missing_cities if missing_cities else 'none'}")
    print(f"\nTotal coverage: {len(df_pop)} / {30 * 12} = {len(df_pop) / 360:.1%}")
    print("\nRows per year:")
    print(df_pop.groupby("year").size())

## GDP per capita (NUTS-3 regional accounts)

Dataset `nama_10r_3gdp`, unit `EUR_HAB` (euro per inhabitant). Queried for all 30 NUTS-3 codes.

Note: Amsterdam uses NUTS-2 (NL32), Rotterdam NUTS-3 (NL33C) may lack data.

In [ ]:
time.sleep(SLEEP_SECONDS)

gdp_params = [("unit", "EUR_HAB"),
              ("sinceTimePeriod", str(YEARS[0])),
              ("untilTimePeriod", str(YEARS[-1]))]
for city_name in CITY_CODES:
    gdp_params.append(("geo", CITY_CODES[city_name]["nuts3"]))

gdp_raw = fetch_eurostat("nama_10r_3gdp", gdp_params)

if gdp_raw:
    df_gdp = parse_jsonstat(gdp_raw, geo_dim="geo")
    df_gdp["city"] = df_gdp["geo"].map(nuts_to_city)
    df_gdp["year"] = df_gdp["time"].astype(int)
    df_gdp = df_gdp.rename(columns={"value": "gdp_per_capita_eur"})
    df_gdp = df_gdp[["city", "year", "gdp_per_capita_eur"]].copy()
    print(f"GDP data: {len(df_gdp)} observations")
    print(f"Cities with data: {df_gdp['city'].nunique()}/30")
    print(f"Years covered: {df_gdp['year'].min()}-{df_gdp['year'].max()}")
else:
    print("ERROR: GDP query failed.")
    df_gdp = pd.DataFrame(columns=["city", "year", "gdp_per_capita_eur"])

In [ ]:
# GDP coverage check
if len(df_gdp) > 0:
    missing_gdp = set(cities["city"]) - set(df_gdp["city"])
    print(f"Cities with ZERO GDP data: {missing_gdp if missing_gdp else 'none'}")
    print(f"\nTotal coverage: {len(df_gdp)} / {30 * 12} = {len(df_gdp) / 360:.1%}")
    print("\nRows per year:")
    print(df_gdp.groupby("year").size())

## Population density (NUTS-3)

Dataset `demo_r_d3dens` provides population density (persons/km²) at NUTS-3 level. We take the latest available year for each city and treat it as a cross-sectional variable.

In [ ]:
time.sleep(SLEEP_SECONDS)

dens_params = [("sinceTimePeriod", str(YEARS[0])),
               ("untilTimePeriod", str(YEARS[-1]))]
for city_name in CITY_CODES:
    dens_params.append(("geo", CITY_CODES[city_name]["nuts3"]))

dens_raw = fetch_eurostat("demo_r_d3dens", dens_params)

if dens_raw:
    df_dens = parse_jsonstat(dens_raw, geo_dim="geo")
    df_dens["city"] = df_dens["geo"].map(nuts_to_city)
    df_dens["year"] = df_dens["time"].astype(int)
    df_dens = df_dens.rename(columns={"value": "population_density"})

    # Take latest available year per city
    df_dens_latest = (
        df_dens.sort_values("year", ascending=False)
        .drop_duplicates(subset="city", keep="first")
        [["city", "population_density"]]
        .copy()
    )
    print(f"Density data: {len(df_dens_latest)} cities (latest year per city)")
    print(df_dens_latest.sort_values("population_density", ascending=False).to_string(index=False))
else:
    print("ERROR: Density query failed.")
    df_dens_latest = pd.DataFrame(columns=["city", "population_density"])

## Assemble the full panel

Start with a complete 30 cities x 12 years grid, then left-join each variable. Cross-sectional variables (density) are merged on city only and repeated across all years.

In [ ]:
# Create the full grid
grid = pd.MultiIndex.from_product(
    [cities["city"].tolist(), YEARS],
    names=["city", "year"]
).to_frame(index=False)

print(f"Full grid: {len(grid)} rows (expected {30 * 12})")

# Merge population (panel)
df_eurostat = grid.merge(df_pop, on=["city", "year"], how="left")

# Merge GDP (panel)
df_eurostat = df_eurostat.merge(df_gdp, on=["city", "year"], how="left")

# Merge density (cross-sectional, on city only)
df_eurostat = df_eurostat.merge(df_dens_latest, on="city", how="left")

print(f"\nAssembled panel: {df_eurostat.shape}")
print(f"Columns: {df_eurostat.columns.tolist()}")
df_eurostat.head()

## Validation

In [ ]:
print(f"Expected 360 rows, got {len(df_eurostat)}")
assert len(df_eurostat) == 360, f"Row count mismatch: expected 360, got {len(df_eurostat)}"

# Check merge key matches cities.csv
eurostat_cities = set(df_eurostat["city"])
cities_set = set(cities["city"])
if eurostat_cities == cities_set:
    print("OK: City names match cities.csv exactly.")
else:
    print(f"MISMATCH: {eurostat_cities.symmetric_difference(cities_set)}")

# Coverage per variable
print("\nNon-null coverage per variable:")
for col in ["population", "gdp_per_capita_eur", "population_density"]:
    n = df_eurostat[col].notna().sum()
    pct = n / len(df_eurostat)
    print(f"  {col}: {n}/{len(df_eurostat)} ({pct:.1%})")

# Cities with zero data
for col in ["population", "gdp_per_capita_eur"]:
    no_data = (
        df_eurostat.groupby("city")[col]
        .apply(lambda x: x.notna().sum())
    )
    missing = no_data[no_data == 0].index.tolist()
    if missing:
        print(f"\nWARNING: Cities with no {col} data at all: {missing}")

# Summary statistics
print("\nSummary statistics:")
print(df_eurostat[["population", "gdp_per_capita_eur", "population_density"]].describe().round(1))

## Quick sanity-check plots

In [ ]:
# Population over time for a few cities
sample_cities = ["Paris", "Berlin", "Madrid", "Warsaw", "Stockholm", "Athens"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Population trends
ax = axes[0]
for c in sample_cities:
    sub = df_eurostat[df_eurostat["city"] == c].dropna(subset=["population"])
    if len(sub) > 0:
        ax.plot(sub["year"], sub["population"] / 1e6, marker="o", markersize=3, label=c)
ax.set_xlabel("Year")
ax.set_ylabel("Population (millions)")
ax.set_title("Population over time")
ax.legend(fontsize=8)

# GDP trends
ax = axes[1]
for c in sample_cities:
    sub = df_eurostat[df_eurostat["city"] == c].dropna(subset=["gdp_per_capita_eur"])
    if len(sub) > 0:
        ax.plot(sub["year"], sub["gdp_per_capita_eur"] / 1000, marker="o", markersize=3, label=c)
ax.set_xlabel("Year")
ax.set_ylabel("GDP per capita (thousand EUR)")
ax.set_title("GDP per capita over time")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Save to CSV

In [ ]:
output_path = "../data/processed/eurostat_city_panel.csv"
df_eurostat.to_csv(output_path, index=False)
print(f"Saved {len(df_eurostat)} rows x {df_eurostat.shape[1]} columns to {output_path}")

## Data gaps and limitations

- **Urban Audit population** lags 1-2 years behind current. Years 2023-2024 may be missing for many cities. Some cities (e.g., Oslo) only have scattered years.
- **GDP per capita** is published at NUTS-3 level. For the Netherlands, NUTS-3 GDP data is largely unavailable; Amsterdam uses NUTS-2 (Noord-Holland), and Rotterdam may have no GDP data at all.
- **Norway (Oslo)** is EEA/EFTA, not EU. Coverage in Eurostat is partial -- population data is sparse, but GDP at NUTS-3 (NO081) is available through ~2021.
- **NUTS code revisions**: The 2021 NUTS revision changed several codes. The mapping above uses current codes, but some time series may break at the revision boundary.
- **Population density** is cross-sectional (latest available year), not a time series. This means density is treated as a static characteristic of each city.
- **Urban Audit boundary changes**: Some cities show discontinuities in population (e.g., Madrid jumps from ~3.2M to ~4.9M in 2015) due to changes in the "greater city" boundary definition, not actual population growth. Downstream analysis should be aware of these breaks.